In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'ultralytics', '--quiet'], check=True)
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import os, yaml, json, shutil
from pathlib import Path
import torch

RDD_DIR  = Path('/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT')
LDMG_DIR = Path('/kaggle/input/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes')

BASE_DIR   = Path('/kaggle/working/road_guard')
OUTPUT_DIR = BASE_DIR / 'output'
EXPORT_DIR = BASE_DIR / 'exports'

for d in [BASE_DIR, OUTPUT_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('RDD train images:', len(list((RDD_DIR / 'train' / 'images').glob('*.jpg'))))
print('RDD val   images:', len(list((RDD_DIR / 'val'   / 'images').glob('*.jpg'))))
print('Dirs ready.')

In [ ]:
CLASS_NAMES = ['longitudinal_crack', 'transverse_crack', 'alligator_crack', 'pothole', 'manhole']

dataset_cfg = {
    'path' : str(RDD_DIR),
    'train': 'train/images',
    'val'  : 'val/images',
    'test' : 'test/images',
    'nc'   : len(CLASS_NAMES),
    'names': CLASS_NAMES,
}

yaml_path = BASE_DIR / 'dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False, allow_unicode=True)

print(open(yaml_path).read())

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')

results = model.train(
    data         = str(yaml_path),
    epochs       = 50,
    imgsz        = 640,
    batch        = 32,
    device       = 0 if torch.cuda.is_available() else 'cpu',
    project      = str(OUTPUT_DIR),
    name         = 'raqib_v1',
    exist_ok     = True,
    patience     = 10,
    save         = True,
    plots        = True,
    workers      = 4,
    cache        = True,
    amp          = True,
    lr0          = 0.01,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    degrees      = 5.0,
    translate    = 0.1,
    scale        = 0.5,
    flipud       = 0.0,
    fliplr       = 0.5,
    mosaic       = 1.0,
)
print('Training done.')

In [ ]:
best_pt = OUTPUT_DIR / 'raqib_v1' / 'weights' / 'best.pt'
print('best.pt exists:', best_pt.exists())

trained_model = YOLO(str(best_pt))

val_results = trained_model.val(
    data   = str(yaml_path),
    imgsz  = 640,
    batch  = 32,
    device = 0 if torch.cuda.is_available() else 'cpu',
)
map50   = float(val_results.box.map50)
map5095 = float(val_results.box.map)
print(f'mAP@50   : {map50:.4f}')
print(f'mAP@50-95: {map5095:.4f}')

In [ ]:
shutil.copy2(best_pt, EXPORT_DIR / 'raqib_pothole_best.pt')
print('Saved .pt')

trained_model.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
onnx_src = best_pt.with_suffix('.onnx')
if onnx_src.exists():
    shutil.copy2(onnx_src, EXPORT_DIR / 'raqib_pothole_best.onnx')
    print('Saved .onnx')

metadata = {
    'model_name'           : 'raqib_pothole_detector_v1',
    'architecture'         : 'YOLOv8m',
    'input_size'           : [640, 640],
    'num_classes'          : len(CLASS_NAMES),
    'class_names'          : CLASS_NAMES,
    'confidence_threshold' : 0.35,
    'iou_threshold'        : 0.45,
    'mAP50'                : round(map50,   4),
    'mAP50_95'             : round(map5095, 4),
    'input_format'         : 'BGR',
    'normalization'        : 'divide_by_255',
}
with open(EXPORT_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print('Saved metadata.json')

print('\nFinal exports:')
for f in sorted(EXPORT_DIR.iterdir()):
    print(f'  {f.name}   {f.stat().st_size/1024/1024:.1f} MB')